In [1]:
%pip install pypdf python-docx beautifulsoup4 lxml


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from isaacus import Isaacus
from dotenv import load_dotenv
import os
import requests
from examples import EXAMPLE_DOCUMENTS
from custom_taxonomy import CUSTOM_TAXONOMY

In [3]:
load_dotenv()  # Load environment variables from `.env` files if any are present.
isaacus_client = Isaacus(api_key=os.getenv("ISAACUS_API_KEY"))

TAXONOMY

TITLE | CLIENT | DATE 

In [4]:
def clean_text(text):
    return text.replace("\n", " ").strip() if text else None

def extract_date(doc):
    for date in doc.dates:
        if date.type == "creation":
            return date.value
    return None

def extract_title(doc):
    return clean_text(doc.title.decode(doc.text)) if doc.title else None

def extract_parties(doc):
    parties = []
    for entity in doc.persons:
        if entity.parent is None and entity.role != "non_party":
            entity_name = clean_text(entity.name.decode(doc.text))
            entity_role = entity.role.replace("_", " ").capitalize() 
            entity_data = {"name": entity_name, "role": entity_role}
            parties.append(entity_data)

    return parties
def extract_locations(doc):
    locations = []
    for entity in doc.locations:
        entity_name = clean_text(entity.name.decode(doc.text))
        locations.append(entity_name)

    return locations
def extract_terms(doc):
    terms = []
    for term in doc.terms:
        term_name = clean_text(term.name.decode(doc.text))
        term_definition = clean_text(term.meaning.decode(doc.text))
        terms.append({"name": term_name, "definition": term_definition})

    return terms

def extradct_signatures(doc):
    signatures = []
    for segment in doc.segments:
        if segment.type == "signature":
            signatures.append(segment.span.decode(doc.text))
    return signatures

def extract_enriched_features_from_document(document_text):
    response = isaacus_client.enrichments.create(
        model="kanon-2-enricher",
        texts=[document_text],
        overflow_strategy="auto",
    )
    doc = response.results[0].document
    features = {}
    features["title"] = extract_title(doc)
    features["parties"] = extract_parties(doc)
    features["date"] = extract_date(doc)
    features["locations"] = extract_locations(doc)
    features["terms"] = extract_terms(doc)
    features["signatures"] = extradct_signatures(doc)
    return features, doc


In [5]:

def score_document_by_category(document_text, query):
    result = isaacus_client.rerankings.create(
        model="kanon-2-reranker",
        query=query,
        texts=[document_text],
    )
    return result.results[0].score

def classify_document(document, nodes, mode="full"):
    if mode == "greedy":
        return classify_greedy(document, nodes)
    if mode == "full":
        return classify_full(document, nodes)

def classify_greedy(document: str, nodes: list[dict]) -> dict:
    scored_nodes = [{
            "node": node,
            "score": score_document_by_category(document, node["description"])
        }
        for node in nodes
    ]

    best = max(scored_nodes, key=lambda item: item["score"])
    best_node = best["node"]

    if not best_node.get("children"):
        return {
            "label": best_node["name"],
            "score": best["score"],
            "description": best_node["description"],
            "mode": "greedy",
        }

    return classify_greedy(document, best_node["children"])


def flatten_taxonomy(nodes):
    all_nodes = []

    for node in nodes:
        all_nodes.append(node)
        children = node.get("children", [])
        if children:
            all_nodes.extend(flatten_taxonomy(children))

    return all_nodes


def classify_full(document, nodes):
    all_nodes = flatten_taxonomy(nodes)

    scored_nodes = [
        {
            "node": node,
            "score": score_document_by_category(document, node["description"]),
        }
        for node in all_nodes
    ]

    best = max(scored_nodes, key=lambda item: item["score"])
    best_node = best["node"]

    return {
        "label": best_node["name"],
        "score": best["score"],
        "description": best_node["description"],
        "mode": "full",
    }



In [6]:
def pretty_title(label, parties, date):
    year = date.split("-")[0] if date else None
    if parties and date:
        pretty_title = f"{label} - {parties[0]} ({year})"
    elif parties:
        pretty_title = f"{label} - {parties[0]}"
    elif date:
        pretty_title= f"{label} ({year[:-2]})"
    return pretty_title


In [7]:
def extract_metadata_from_text(document_text, taxonomy_mode="full"):
    features, doc = extract_enriched_features_from_document(document_text)
    category = classify_document(document_text, CUSTOM_TAXONOMY, mode=taxonomy_mode)
    features["category"] = category
    features["pretty_title"] = pretty_title(category["label"], features["parties"], features["date"])
    print(features)


In [8]:
for document in EXAMPLE_DOCUMENTS:
    break
    extract_metadata_from_text(document, taxonomy_mode="greedy")

In [9]:
import io
import base64
import mimetypes
import time
import traceback
from pathlib import Path

import ipywidgets as widgets
from pypdf import PdfReader
from docx import Document as DocxDocument
from bs4 import BeautifulSoup
from ilgs_viewer_module import render_viewer
def flatten_leaf_labels(nodes):
    labels = []
    for node in nodes:
        children = node.get("children", [])
        if children:
            labels.extend(flatten_leaf_labels(children))
        else:
            labels.append(node["name"])
    return labels

CATEGORY_OPTIONS = sorted(set(flatten_leaf_labels(CUSTOM_TAXONOMY)))

def uploaded_items(value):
    if isinstance(value, dict):
        return list(value.values())
    if isinstance(value, (list, tuple)):
        return list(value)
    return []

def to_bytes(content):
    if isinstance(content, bytes):
        return content
    if isinstance(content, memoryview):
        return content.tobytes()
    if isinstance(content, bytearray):
        return bytes(content)
    raise TypeError(f"Unsupported file content type: {type(content)}")

def extract_text_from_pdf(file_bytes):
    reader = PdfReader(io.BytesIO(file_bytes))
    return "\\n\\n".join((page.extract_text() or "") for page in reader.pages).strip()

def extract_text_from_docx(file_bytes):
    doc = DocxDocument(io.BytesIO(file_bytes))
    return "\\n\\n".join(p.text for p in doc.paragraphs if p.text and p.text.strip()).strip()

def extract_text_from_html(file_bytes):
    text = file_bytes.decode("utf-8", errors="ignore")
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text("\\n", strip=True)

def extract_text_from_plain(file_bytes):
    try:
        return file_bytes.decode("utf-8")
    except UnicodeDecodeError:
        return file_bytes.decode("latin-1", errors="ignore")

def bytes_to_text(file_bytes, name=""):
    suffix = Path(name).suffix.lower()
    if suffix == ".pdf":
        return extract_text_from_pdf(file_bytes)
    if suffix == ".docx":
        return extract_text_from_docx(file_bytes)
    if suffix in {".html", ".htm"}:
        return extract_text_from_html(file_bytes)
    return extract_text_from_plain(file_bytes)

def make_preview_payload(file_bytes, name):
    mime_type = mimetypes.guess_type(name)[0] or "application/octet-stream"
    preview_b64 = None

    # Larger threshold so normal PDFs can still preview in the UI.
    # If notebook performance becomes an issue, reduce this again.
    if mime_type == "application/pdf" and len(file_bytes) <= 5_000_000:
        preview_b64 = base64.b64encode(file_bytes).decode("ascii")

    return mime_type, preview_b64

def ingest_file(file_info, taxonomy_mode="full"):
    name = file_info.get("name", "Untitled")
    raw_content = file_info.get("content", b"")
    file_bytes = to_bytes(raw_content)
    document_text = bytes_to_text(file_bytes, name=name)

    if not document_text or len(document_text.strip()) < 20:
        raise ValueError("Very little text was extracted. If this is a scanned PDF, you may need OCR.")

    document_text = document_text[:80000]

    features, enriched_doc = extract_enriched_features_from_document(document_text)
    category = classify_document(document_text, CUSTOM_TAXONOMY, mode=taxonomy_mode)
    mime_type, preview_b64 = make_preview_payload(file_bytes, name)

    parties = features.get("parties", [])
    party_names = [p["name"] for p in parties if isinstance(p, dict) and p.get("name")]

    return {
        "source_name": name,
        "title": features.get("title"),
        "pretty_title": pretty_title(category["label"], party_names, features.get("date")),
        "date": features.get("date"),
        "parties": parties,
        "locations": features.get("locations", []),
        "terms": features.get("terms", []),
        "signatures": features.get("signatures", []),
        "category": category,
        "_mime_type": mime_type,
        "_preview_base64": preview_b64,
        "_text_excerpt": document_text[:6000],
        "enriched_doc": enriched_doc,
    }

records = []

In [10]:
upload = widgets.FileUpload(
    accept=".txt,.md,.html,.pdf,.docx,.doc",
    multiple=True,
)

file_progress = widgets.IntProgress(
    value=0,
    min=0,
    max=1,
    description="Files",
    bar_style="",
    layout=widgets.Layout(width="220px"),
)

stage_progress = widgets.IntProgress(
    value=0,
    min=0,
    max=3,
    description="Stage",
    bar_style="",
    layout=widgets.Layout(width="220px"),
)

status = widgets.HTML("<b>Status:</b> Ready")
stage_label = widgets.HTML("<b>Stage:</b> Idle")
debug = widgets.Output(layout={"border": "1px solid #e5e7eb", "padding": "8px", "max_height": "160px", "overflow": "auto"})
viewer_out = widgets.Output()

def log(*args):
    with debug:
        print(*args)

def set_stage(step, message):
    stage_progress.value = step
    stage_label.value = f"<b>Stage:</b> {message}"

def refresh_viewer():
    render_viewer(
        records,
        category_options=CATEGORY_OPTIONS,
        status=status,
        stage_label=stage_label,
        viewer_out=viewer_out,
        log=log,
    )

def on_upload(change):
    new_files = uploaded_items(change["new"])
    total = len(new_files)

    file_progress.max = max(total, 1)
    file_progress.value = 0
    file_progress.bar_style = "info"

    stage_progress.max = 3
    stage_progress.value = 0
    stage_progress.bar_style = "info"

    status.value = f"<b>Status:</b> Ingesting {total} file(s)…"

    try:
        for i, file_info in enumerate(new_files, start=1):
            set_stage(1, f"Reading {file_info.get('name', 'file')}")
            record = ingest_file(file_info, taxonomy_mode="full")
            set_stage(2, "Appending record")
            records.append(record)
            file_progress.value = i

        set_stage(3, "Rendering viewer")
        refresh_viewer()

        file_progress.bar_style = "success"
        stage_progress.bar_style = "success"
        status.value = f"<b>Status:</b> Ready · {len(records)} document(s)"
    except Exception as e:
        file_progress.bar_style = "danger"
        stage_progress.bar_style = "danger"
        status.value = f"<b>Status:</b> Error: {e}"
        log(traceback.format_exc())

    try:
        upload.value = ()
        upload._counter = 0
    except Exception:
        pass

try:
    upload.unobserve_all()
except Exception:
    pass

upload.observe(on_upload, names="value")

toolbar = widgets.HBox(
    [
        widgets.HTML(
            "<div style='display:flex;flex-direction:column;gap:2px;'>"
            "<div style='font:600 15px/1.2 Inter,-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif;'>Metadata Viewer</div>"
            "<div style='font:12px/1.2 Inter,-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif;color:#6d7b90;'>Upload, enrich, classify, and review documents.</div>"
            "</div>"
        ),
        widgets.HBox(
            [file_progress, stage_progress, upload],
            layout=widgets.Layout(gap="10px", align_items="center"),
        ),
    ],
    layout=widgets.Layout(justify_content="space-between", align_items="center", width="100%"),
)

display(toolbar)
display(status)
display(stage_label)
display(debug)
display(viewer_out)

refresh_viewer()

HTML(value='<b>Status:</b> Ready')

HTML(value='<b>Stage:</b> Idle')

Output(layout=Layout(border_bottom='1px solid #e5e7eb', border_left='1px solid #e5e7eb', border_right='1px sol…

Output()